In [ ]:
# --- Cell 1: Upload Dataset ---
from google.colab import files
import pandas as pd

print("Upload your dataset:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Dataset Loaded:", df.shape)
df.head()

In [ ]:
# --- Cell 2: Create Target (processing_days) ---
df['case_submitted'] = pd.to_datetime(df['case_submitted'])
df['decision_date']  = pd.to_datetime(df['decision_date'])

df['processing_days'] = (df['decision_date'] - df['case_submitted']).dt.days
df = df[df['processing_days'] > 0]

print("Target created! processing_days stats:")
print(df['processing_days'].describe())

In [ ]:
# --- Cell 3: Define Features and Target ---
DROP_COLS = ['processing_days', 'case_submitted', 'decision_date',
             'emp_name', 'lat', 'lng']

X = df.drop(columns=DROP_COLS, errors='ignore').copy()
y = df['processing_days']

# Save the exact feature order — app.py must match this
FEATURE_NAMES = list(X.columns)
print("Features:", FEATURE_NAMES)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# --- Cell 4: Encode Categorical Data & SAVE the encoders ---
# FIX: store every LabelEncoder so inference can reproduce the same mapping.
from sklearn.preprocessing import LabelEncoder

label_encoders = {}   # col -> fitted LabelEncoder

for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        # Normalise to upper-case strings so inference matches training
        X[col] = le.fit_transform(X[col].fillna('UNKNOWN').astype(str).str.upper())
        label_encoders[col] = le   # <-- SAVE IT
        print(f"  {col:30s}  {len(le.classes_):>6,} unique values")

print(f"\nEncoding done! Saved {len(label_encoders)} encoders.")

In [ ]:
# --- Cell 5: Train-Test Split ---
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "  Test:", X_test.shape)

In [ ]:
# --- Cell 6: Train XGBoost (GPU) ---
!pip install xgboost -q

from xgboost import XGBRegressor

model = XGBRegressor(
    tree_method='hist',
    device='cuda',
    n_estimators=150,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete!")

In [ ]:
# --- Cell 7: Evaluate Model ---
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n🔥 XGBoost GPU Model Performance:")
print(f"MAE:  {mae:.2f} days")
print(f"RMSE: {rmse:.2f} days")
print(f"R²:   {r2:.4f}")

In [ ]:
# --- Cell 8: Save ALL artifacts & download ---
import pickle, zlib
from google.colab import files

# 1. Model — zlib-compressed (matches what app.py expects)
with open('xgboost_visa_model.pkl', 'wb') as f:
    f.write(zlib.compress(pickle.dumps(model)))

# 2. LabelEncoders — the missing piece!
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# 3. Feature list in exact training order
with open('features.pkl', 'wb') as f:
    pickle.dump(FEATURE_NAMES, f)

print("Saved: xgboost_visa_model.pkl, label_encoders.pkl, features.pkl")
print("Downloading all three...")

files.download('xgboost_visa_model.pkl')
files.download('label_encoders.pkl')
files.download('features.pkl')